In [3]:
# 새 데이터셋 — '옷장패션' 주문 (가상)
import numpy as np
import pandas as pd

np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

# 오염 심기
# (a) 나이 이상치 — 입력 실수(0, 999)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0

# (b) amount 결측 — app 채널에 더 자주 (MAR 시그널)
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan

# (c) return_amount 결측은 그대로 (0=환불없음)이라 결측 아님. 단, '관찰 안 됨'을 의도적으로 표현하기 위해
#     price 결측 5건 추가(접속 시점 가격이 누락된 사례)
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan

# (d) quantity 이상치(단일 소비자 200개)
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200

# (e) amount 극단값(50,000,000짜리 한 건 — '도매 의심')
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)
partner.head()

옷장패션 데이터 준비 완료: (1500, 8)


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
2,K00003,29,상의,web,49900.0,2,99800.0,0.0
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
4,K00005,33,하의,app,129900.0,1,129900.0,0.0


In [4]:
partner.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       1500 non-null   str    
 1   customer_age   1500 non-null   int64  
 2   category       1500 non-null   str    
 3   channel        1500 non-null   str    
 4   price          1495 non-null   float64
 5   quantity       1500 non-null   int64  
 6   amount         1449 non-null   float64
 7   return_amount  1500 non-null   float64
dtypes: float64(3), int64(2), str(3)
memory usage: 93.9 KB


In [14]:
partner[partner['price'].isna()]

,order_id,customer_age,category,channel,price,quantity,amount,return_amount
656,K00657,26,신발,app,NaN,3,149700.0,0.0
669,K00670,43,하의,web,NaN,1,79900.0,0.0
822,K00823,52,하의,web,NaN,1,15900.0,0.0
931,K00932,39,액세서리,web,NaN,1,129900.0,100018.0
1187,K01188,12,상의,app,NaN,2,59800.0,0.0


In [28]:
print(partner["order_id"].count())

1500


In [29]:
print(partner["amount"].count())

1449


In [39]:
print(100 - partner["amount"].count() / partner["order_id"].count() * 100)

3.4000000000000057


In [40]:
print(100 - partner["price"].count() / partner["order_id"].count() * 100)

0.3333333333333286


In [75]:
q1 = partner["quantity"].quantile(0.25)
q3 = partner["quantity"].quantile(0.75)
iqr = q3 - q1

print(q1)
print(q3)
print(q3-q1)

1.0
2.0
1.0


In [76]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(lower)
print(upper)

-0.5
3.5


In [80]:
outlier_quantity = (partner["quantity"] < lower) | (partner["quantity"] > upper)
outliers = partner[outlier_quantity]

print(outliers["quantity"].unique())
display(outliers)

[200]


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
24,K00025,38,신발,app,29900.0,200,89700.0,0.0


In [81]:
partner.loc[partner['quantity'] == 200]

,order_id,customer_age,category,channel,price,quantity,amount,return_amount
24,K00025,38,신발,app,29900.0,200,89700.0,0.0


In [79]:
partner.loc[partner['quantity'] == 0]

,order_id,customer_age,category,channel,price,quantity,amount,return_amount
719,K00720,0,하의,web,129900.0,2,259800.0,0.0
1436,K01437,0,하의,web,15900.0,2,31800.0,0.0
